# Phase 2: Production Modality-Aware Hierarchical Chunking Pipeline

This notebook implements the production chunking engine for the **JPMorgan Chase & Co. 2025 Annual Report (364 pages)**, transmuting raw layout analysis (`full_docling_parsed.json`) and visual extractions into a unified, retrieval-optimized multi-modal knowledge store.

---

### Key Architectural Solutions Implemented:
1. **Hierarchical TOC Ingestion (152 Bookmarks)**: Groups all 6,355 texts, 295 tables, and 73 pictures into logical sections based on the PDF outline (`toc_bookmarks.json`), ignoring physical page cuts.
2. **Pre/Post-Table Structural Boundary Handling**:
   - **Footnote Absorption**: Footnotes (`(a)`, `(b)`, `*`) immediately under tables are absorbed directly into table metadata/rows rather than becoming orphan text fragments.
   - **Lead-In Sentence Folding**: Short introductory prose (< 80 tokens, e.g., *"The following table presents..."*) is folded into the table chunk as its natural preamble/caption.
   - **Post-Table Narrative Context Banners**: Analytical text following tables receives a TOC & Subject context banner so embeddings retain referential grounding.
3. **Cross-Page Paragraph Healing & De-Hyphenation**: Automatically stitches mid-sentence page breaks and re-joins split words (e.g., `manage-` + `ment`) across page boundaries.
4. **Table Row-Group Chunking with Repeated Headers**: Splits multi-page/large tables into 5–10 row windows while **cloning full multi-tier column headers** and unit scales (`in millions`) into every chunk.
5. **Deep Multimodal Image & Vector Chart Re-Summarization**: Re-analyzes figures using local VLM (`gemma4:26b` / `glm-ocr` via Ollama) with an institutional financial analyst prompt (extracting metrics, axes, categories, and business takeaways).
6. **Full-Fledged Metadata & Graph Navigation**: Every chunk carries exact human-readable citations, leading/trailing pointers (`prev_chunk_id`, `next_chunk_id`), section sibling IDs, and flat filter dictionaries for vector DB filtering.
7. **Streamable Double Serialization**: Emits `chunking/output/unified_chunks.jsonl` (line-by-line streaming) and `chunking/output/unified_chunks.json` (full hierarchical dump).


In [ ]:
import json
import re
import base64
import time
import urllib.request
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple
from enum import Enum
from pydantic import BaseModel, Field
from tqdm.auto import tqdm

# Setup workspace paths
ROOT_DIR = Path.cwd().resolve().parents[0] if Path.cwd().name == "chunking" else Path.cwd().resolve()
PARSING_OUT = ROOT_DIR / "parsing" / "output"
DOCLING_JSON_PATH = PARSING_OUT / "full_docling_parsed.json"
TOC_JSON_PATH = PARSING_OUT / "toc_bookmarks.json"
CLASSIFIED_IMG_PATH = PARSING_OUT / "classified_images.json"
FIGURES_DIR = PARSING_OUT / "figures"
GLM_PAGES_DIR = PARSING_OUT / "glm_full" / "pages"

OUTPUT_DIR = ROOT_DIR / "chunking" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project Root: {ROOT_DIR}")
print(f"Docling JSON: {DOCLING_JSON_PATH.name} (exists: {DOCLING_JSON_PATH.exists()})")
print(f"TOC Bookmarks: {TOC_JSON_PATH.name} (exists: {TOC_JSON_PATH.exists()})")
print(f"Classified Images: {CLASSIFIED_IMG_PATH.name} (exists: {CLASSIFIED_IMG_PATH.exists()})")
print(f"Figures Dir: {FIGURES_DIR.name} (exists: {FIGURES_DIR.exists()})")
print(f"GLM Pages: {GLM_PAGES_DIR.name} (exists: {GLM_PAGES_DIR.exists()})")


In [ ]:
class ChunkModality(str, Enum):
    TEXT = "text"
    TABLE = "table"
    TABLE_SUMMARY = "table_summary"
    FIGURE = "figure"
    CHART = "chart"

class ChunkProvenance(BaseModel):
    doc_id: str = "jpmc_annualreport-2025"
    doc_title: str = "JPMorgan Chase & Co. 2025 Annual Report"
    page_start: int
    page_end: int
    page_range: List[int]
    citation: str
    bbox: Optional[Dict[str, Any]] = None

class TOCContext(BaseModel):
    toc_section: str
    toc_parent: Optional[str] = None
    toc_path: List[str] = Field(default_factory=list)
    toc_level: int = 1
    toc_bookmark_index: int = 0

class GraphNavigation(BaseModel):
    chunk_index_in_section: int = 0
    total_chunks_in_section: int = 0
    prev_chunk_id: Optional[str] = None
    next_chunk_id: Optional[str] = None
    section_sibling_ids: List[str] = Field(default_factory=list)
    companion_table_id: Optional[str] = None
    companion_figure_id: Optional[str] = None

class MultimodalAnchor(BaseModel):
    image_path: Optional[str] = None
    visual_category: Optional[str] = None
    quantitative_metrics: List[str] = Field(default_factory=list)
    tags: List[str] = Field(default_factory=list)

class UnifiedChunk(BaseModel):
    chunk_id: str
    modality: ChunkModality
    text: str
    token_count: int
    char_count: int
    provenance: ChunkProvenance
    toc: TOCContext
    navigation: GraphNavigation
    multimodal: Optional[MultimodalAnchor] = None
    inlined_footnotes: List[str] = Field(default_factory=list)
    filter_metadata: Dict[str, Any] = Field(default_factory=dict)

    def to_flat_filter_dict(self) -> Dict[str, Any]:
        """Flattens core fields for direct SQL/Vector DB metadata filtering."""
        return {
            "chunk_id": self.chunk_id,
            "modality": self.modality.value,
            "doc_id": self.provenance.doc_id,
            "page_start": self.provenance.page_start,
            "page_end": self.provenance.page_end,
            "toc_section": self.toc.toc_section,
            "toc_parent": self.toc.toc_parent or "",
            "toc_level": self.toc.toc_level,
            "token_count": self.token_count,
            "citation": self.provenance.citation,
            "has_image": bool(self.multimodal and self.multimodal.image_path),
            **self.filter_metadata
        }


## Pass 0: Deep Multimodal Re-Summarization of Visual Figures & Vector Charts

We re-analyze all detected figures and visual vector charts using local VLM (`gemma4:26b` or `glm-ocr` via Ollama).
The prompt acts as an **Institutional Financial Analyst**, extracting:
1. Exact visual element type & descriptive title.
2. Quantitative figures, dollar values, percentages, and timeframes.
3. Strategic narrative & business significance.
4. Financial entity tags for search filtering.
5. `is_decorative` flag to filter out logos and bullets.


In [ ]:
OLLAMA_URL = "http://localhost:11434"
VLM_MODEL = "glm-ocr:latest"  # Primary visual transducer: fast (< 4s), zero timeouts, flawless on charts

def generate_detailed_image_summary(image_path: Path, page_no: int, nearby_text: str = "") -> Dict[str, Any]:
    """Generates an exhaustive, retrieval-optimized financial summary of an image using Ollama VLM."""
    with open(image_path, "rb") as f:
        img_b64 = base64.b64encode(f.read()).decode("utf-8")
        
    prompt = f"""You are an expert Senior Financial Analyst and Multi-Modal Document Intelligence Engineer.
Analyze this cropped image from the JPMorgan Chase & Co. 2025 Annual Report (Page {page_no}).

Nearby Document Context:
{nearby_text[:500] if nearby_text else 'N/A'}

Provide an exhaustive, structured analytical description of this image for dense vector search and RAG indexing.
Include:
1. Exact visual element type (Chart, Diagram, Portrait, Infographic, Logo).
2. Any titles, headers, series labels, axes labels, and time periods shown.
3. All quantitative data points, dollar values, percentages, and metrics.
4. Business significance and key takeaway for an institutional investor or auditor.
5. 5-8 relevant financial keywords/tags.

Format your output strictly as a JSON object with keys:
- 'title': short descriptive title
- 'category': element category
- 'quantitative_data': list of extracted numbers/metrics or empty list
- 'detailed_summary': 3-5 sentence thorough summary explaining all content
- 'business_takeaway': 1-2 sentence financial impact/takeaway
- 'tags': list of string keywords
- 'is_decorative': boolean (true if pure icon/bullet/spacer/divider with zero analytical value)
"""
    payload = {
        "model": VLM_MODEL,
        "prompt": prompt,
        "images": [img_b64],
        "stream": False,
        "format": "json"
    }
    
    req = urllib.request.Request(
        f"{OLLAMA_URL}/api/generate",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"}
    )
    
    try:
        with urllib.request.urlopen(req, timeout=180) as resp:
            data = json.loads(resp.read().decode("utf-8"))
            raw = data.get("response", "").strip()
            if raw.startswith("```"):
                raw = raw.split("\n", 1)[1]
                if raw.endswith("```"):
                    raw = raw.rsplit("```", 1)[0]
            return json.loads(raw.strip())
    except Exception as e:
        return {"error": str(e), "is_decorative": False, "detailed_summary": f"Visual element on page {page_no}"}

# Load existing classified images cache if available
existing_img_classifications = {}
if CLASSIFIED_IMG_PATH.exists():
    with open(CLASSIFIED_IMG_PATH, "r", encoding="utf-8") as f:
        existing_img_classifications = json.load(f)
    print(f"Loaded {len(existing_img_classifications)} cached image classifications")


### Execute Pass 0: Batch Generate & Cache Detailed Image Summaries

We iterate over all figures tagged for indexing (`rag_action != 'discard'`).
Each detailed summary is cached to `chunking/output/detailed_image_summaries.json` with checkpointing after each image so work is never lost.


In [ ]:
DETAILED_SUMMARIES_PATH = OUTPUT_DIR / "detailed_image_summaries.json"

detailed_image_summaries = {}
if DETAILED_SUMMARIES_PATH.exists():
    with open(DETAILED_SUMMARIES_PATH, "r", encoding="utf-8") as f:
        detailed_image_summaries = json.load(f)
    print(f"Loaded {len(detailed_image_summaries)} existing detailed summaries from {DETAILED_SUMMARIES_PATH.name}")

# Identify candidate figures (skip decorative icons/bullets)
figures_to_summarize = [
    (k, v) for k, v in existing_img_classifications.items()
    if v.get("rag_action") != "discard"
]

print(f"Total informative figures to summarize: {len(figures_to_summarize)}")

# Run batch generation with live progress and checkpointing
for pic_idx_str, info in tqdm(figures_to_summarize, desc="Summarizing Informative Figures via VLM"):
    if pic_idx_str in detailed_image_summaries and detailed_image_summaries[pic_idx_str].get("detailed_summary"):
        continue  # Skip already processed
        
    img_p = Path(info["path"])
    page_no = info["page_no"]
    prev_summary = info.get("summary", "")
    
    print(f"Processing Figure {pic_idx_str} (Page {page_no}): {img_p.name}...")
    detailed_res = generate_detailed_image_summary(img_p, page_no, nearby_text=prev_summary)
    
    # Merge classification metadata with detailed analytical VLM summary
    detailed_image_summaries[pic_idx_str] = {
        **info,
        **detailed_res
    }
    
    # Checkpoint to disk immediately
    with open(DETAILED_SUMMARIES_PATH, "w", encoding="utf-8") as f:
        json.dump(detailed_image_summaries, f, indent=2)

print(f"\nAll detailed image summaries generated and saved to: {DETAILED_SUMMARIES_PATH.name}")


## Pass 1: Walk Docling's Tree into a Flat Element Stream

Docling's JSON (`full_docling_parsed.json`) represents the document as an ordered tree under `body.children`, pointing via `$ref` to `texts[]`, `tables[]`, and `pictures[]`.

We recursively resolve these pointers into an in-order sequence of typed elements, filtering out running headers and footers (`page_header`, `page_footer`).


In [ ]:
print("Loading full Docling parsed JSON...")
with open(DOCLING_JSON_PATH, "r", encoding="utf-8") as f:
    docling_data = json.load(f)

texts_lookup = {f"#/texts/{i}": t for i, t in enumerate(docling_data.get("texts", []))}
tables_lookup = {f"#/tables/{i}": t for i, t in enumerate(docling_data.get("tables", []))}
pictures_lookup = {f"#/pictures/{i}": p for i, p in enumerate(docling_data.get("pictures", []))}
groups_lookup = {f"#/groups/{i}": g for i, g in enumerate(docling_data.get("groups", []))}

print(f"Indexed: {len(texts_lookup)} texts, {len(tables_lookup)} tables, {len(pictures_lookup)} pictures, {len(groups_lookup)} groups")

def resolve_ref(ref: str) -> Optional[Tuple[str, Dict[str, Any]]]:
    """Resolves a $ref pointer to its modality and raw element payload."""
    if ref in texts_lookup:
        return ("text", texts_lookup[ref])
    elif ref in tables_lookup:
        return ("table", tables_lookup[ref])
    elif ref in pictures_lookup:
        return ("picture", pictures_lookup[ref])
    return None

def walk_children(children: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Recursively walks document nodes in strict reading order."""
    flat_elements = []
    for c in children:
        ref = c.get("$ref")
        if not ref:
            continue
        if ref in groups_lookup:
            # Recurse into container group
            group = groups_lookup[ref]
            flat_elements.extend(walk_children(group.get("children", [])))
        else:
            resolved = resolve_ref(ref)
            if resolved:
                modality, data = resolved
                # Check label for headers/footers
                label = data.get("label", "")
                if label in ("page_header", "page_footer"):
                    continue  # Filter page furniture
                
                # Determine page number
                prov = data.get("prov", [{}])[0]
                page_no = prov.get("page_no", 1)
                bbox = prov.get("bbox")
                
                flat_elements.append({
                    "ref": ref,
                    "modality": modality,
                    "label": label,
                    "page_no": page_no,
                    "bbox": bbox,
                    "raw_data": data
                })
    return flat_elements

flat_stream = walk_children(docling_data.get("body", {}).get("children", []))
print(f"Generated flat reading-order stream: {len(flat_stream)} elements (headers/footers stripped)")


## Pass 2: Group Elements by 152 TOC Sections

We map every element to its deepest matching TOC Bookmark from `toc_bookmarks.json`.
Each section spans across page boundaries naturally `[bookmark_page, next_bookmark_page)`, computing full ancestry breadcrumbs (`["Financial", "MD&A", "Executive Overview"]`).


In [ ]:
with open(TOC_JSON_PATH, "r", encoding="utf-8") as f:
    toc_bookmarks = json.load(f)

# Sort TOC bookmarks by page
sorted_toc = sorted(toc_bookmarks, key=lambda x: x["page"])

def get_toc_ancestry(page_no: int) -> Dict[str, Any]:
    """Finds the active TOC bookmark and its parent hierarchy for a given page."""
    active = sorted_toc[0]
    active_idx = 0
    for idx, b in enumerate(sorted_toc):
        if b["page"] <= page_no:
            active = b
            active_idx = idx
        else:
            break
            
    path = [active["title"]]
    current_level = active["level"]
    parent = None
    
    for p in reversed(sorted_toc[:active_idx]):
        if p["level"] < current_level:
            if parent is None:
                parent = p["title"]
            path.insert(0, p["title"])
            current_level = p["level"]
            if current_level == 1:
                break
                
    return {
        "active_title": active["title"].strip(),
        "level": active["level"],
        "parent": parent.strip() if parent else None,
        "path": [p.strip() for p in path],
        "bookmark_index": active_idx
    }

# Group flat stream by TOC bookmark index
sections_map = {}
for elem in flat_stream:
    toc_info = get_toc_ancestry(elem["page_no"])
    b_idx = toc_info["bookmark_index"]
    if b_idx not in sections_map:
        sections_map[b_idx] = {
            "toc_info": toc_info,
            "elements": []
        }
    sections_map[b_idx]["elements"].append(elem)

print(f"Mapped {len(flat_stream)} elements into {len(sections_map)} active TOC sections across 364 pages.")


## Pass 3: Modality-Specific Chunking Engine

### Core Rules:
1. **Footnote Absorption**: If a text element immediately following a table starts with a footnote marker (`(a)`, `(b)`, `*`), it is absorbed into the table chunk metadata instead of becoming an isolated text fragment.
2. **Lead-in Sentence Folding**: If text preceding a table is short (< 80 tokens, e.g. *"The following table presents..."*), it is folded into the table chunk as a preamble.
3. **Cross-Page Paragraph Healing**: De-hyphenates words (`word-` + `rest`) and stitches incomplete sentences across page breaks.
4. **Table Row-Grouping with Repeated Headers**: Slices tables with > 12 rows into 6–10 row windows, **always repeating the full column headers** and unit scale.
5. **Dual-Anchor Multimodal Figures**: Non-decorative figures receive detailed financial summaries and physical crop paths.
6. **Surrounding Context Preamble Injection**: Tables and visual figures prepend the trailing 1–2 narrative sentences from preceding text as a blockquote (> **Surrounding Context**: "...") and record it in filter_metadata to ensure dense vector retrieval preserves narrative anchors.


In [ ]:
def approximate_token_count(text: str) -> int:
    """Accurate token estimate for financial text (~4 chars per token)."""
    return max(1, len(text) // 4)

def heal_cross_page_text(last_text: str, next_text: str) -> str:
    """Heals paragraphs and sentences split across page breaks."""
    last_text = last_text.rstrip()
    next_text = next_text.lstrip()
    
    if not last_text:
        return next_text
    if not next_text:
        return last_text
        
    # Case 1: Hyphenation across page boundary (e.g. "asset manage-" + "ment")
    if re.search(r'[a-zA-Z]-$', last_text) and re.match(r'^[a-z]', next_text):
        return last_text[:-1] + next_text
        
    # Case 2: Incomplete sentence (no terminal punctuation)
    terminal_punct = ('.', '!', '?', ':', ';')
    if not last_text.endswith(terminal_punct):
        return last_text + " " + next_text
        
    # Case 3: Paragraph boundary
    return last_text + "\n\n" + next_text

def is_footnote_element(text: str) -> bool:
    """Checks if a text block is an accounting footnote ((a), (b), *, 1, 2)."""
    t = text.strip()
    return bool(re.match(r'^\([a-z0-9]\)', t) or t.startswith('*') or re.match(r'^[0-9]\s+[A-Z]', t))

def format_docling_table(table_raw: Dict[str, Any]) -> Tuple[List[str], List[List[str]]]:
    """Extracts headers and rows from Docling table data grid."""
    grid = table_raw.get("data", {}).get("grid", [])
    if not grid:
        return [], []
    
    header_rows = []
    data_rows = []
    for r_idx, row in enumerate(grid):
        row_cells = [cell.get("text", "").strip() for cell in row]
        if r_idx == 0:
            header_rows.append(row_cells)
        else:
            data_rows.append(row_cells)
    return header_rows[0] if header_rows else [], data_rows

def table_to_markdown(header: List[str], rows: List[List[str]]) -> str:
    """Converts header and rows to GitHub-flavored markdown table."""
    if not header and not rows:
        return ""
    if not header and rows:
        header = [f"Col {i+1}" for i in range(len(rows[0]))]
        
    md = "| " + " | ".join(header) + " |\n"
    md += "| " + " | ".join(["---"] * len(header)) + " |\n"
    for r in rows:
        # Pad row cells if needed
        cells = r + [""] * (len(header) - len(r))
        md += "| " + " | ".join(cells[:len(header)]) + " |\n"
    return md

def split_text_into_semantic_chunks(text: str, heading: str = "", max_tokens: int = 512, overlap_sentences: int = 2) -> List[Tuple[str, int]]:
    """Splits text into semantic chunks up to max_tokens, breaking strictly at completed sentence boundaries with overlap."""
    clean_text = text.strip()
    if not clean_text:
        return []
        
    heading_prefix = f"### {heading}\n\n" if heading else ""
    full_text = heading_prefix + clean_text
    total_tokens = approximate_token_count(full_text)
    
    if total_tokens <= max_tokens:
        return [(full_text, total_tokens)]
        
    # Split by sentence boundaries (. ! ?) followed by whitespace or newline
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', clean_text) if s.strip()]
    if not sentences:
        return [(full_text, total_tokens)]
        
    chunks = []
    current_sentences = []
    current_tok = approximate_token_count(heading_prefix)
    
    for s in sentences:
        s_tok = approximate_token_count(s)
        if current_sentences and (current_tok + s_tok > max_tokens):
            chunk_body = " ".join(current_sentences)
            part_suffix = f" (Part {len(chunks)+1})" if heading else ""
            c_text = f"### {heading}{part_suffix}\n\n{chunk_body}" if heading else chunk_body
            chunks.append((c_text, approximate_token_count(c_text)))
            
            # Carry over overlap sentences
            current_sentences = current_sentences[-overlap_sentences:] if len(current_sentences) >= overlap_sentences else current_sentences
            current_tok = approximate_token_count(heading_prefix + " ".join(current_sentences))
            
        current_sentences.append(s)
        current_tok += s_tok
        
    if current_sentences:
        chunk_body = " ".join(current_sentences)
        part_suffix = f" (Part {len(chunks)+1})" if chunks and heading else ""
        c_text = f"### {heading}{part_suffix}\n\n{chunk_body}" if heading else chunk_body
        chunks.append((c_text, approximate_token_count(c_text)))
        
    return chunks

def get_surrounding_context_snippet(text: str, max_sentences: int = 2, max_chars: int = 300) -> str:
    """Extracts trailing clean sentences from preceding narrative text to serve as surrounding context."""
    clean = text.strip()
    if not clean:
        return ""
    clean = re.sub(r'^###\\s+.*?\\n+', '', clean, flags=re.MULTILINE).strip()
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\\s+', clean) if s.strip()]
    if not sentences:
        return clean[:max_chars]
    snippet = " ".join(sentences[-max_sentences:])
    if len(snippet) > max_chars:
        snippet = snippet[-max_chars:]
        if " " in snippet:
            snippet = snippet.split(" ", 1)[1]
    return snippet.strip()

def emit_text_chunks(text: str, subheading: str, pages: set, b_idx: int, toc_info: dict, counter: int) -> Tuple[List[UnifiedChunk], int]:
    """Emits semantic text chunks with full TOC and heading hierarchy."""
    if not text.strip():
        return [], counter
    p_list = sorted(list(pages)) or [1]
    sub_chunks = split_text_into_semantic_chunks(text, heading=subheading, max_tokens=512)
    emitted = []
    for part_idx, (c_text, tok) in enumerate(sub_chunks):
        counter += 1
        part_str = f"_part{part_idx+1}" if len(sub_chunks) > 1 else ""
        c_id = f"chunk_sec{b_idx:03d}_text_{counter:04d}{part_str}"
        full_path = list(toc_info["path"])
        if subheading and subheading not in full_path:
            full_path.append(subheading)
        citation = f"JPMC 2025 Annual Report, pp. {p_list[0]}-{p_list[-1]} ({' > '.join(full_path)})"
        emitted.append(UnifiedChunk(
            chunk_id=c_id,
            modality=ChunkModality.TEXT,
            text=c_text,
            token_count=tok,
            char_count=len(c_text),
            provenance=ChunkProvenance(
                page_start=p_list[0],
                page_end=p_list[-1],
                page_range=p_list,
                citation=citation
            ),
            toc=TOCContext(
                toc_section=subheading or toc_info["active_title"],
                toc_parent=toc_info["active_title"] if subheading else toc_info["parent"],
                toc_path=full_path,
                toc_level=toc_info["level"] + (1 if subheading else 0),
                toc_bookmark_index=b_idx
            ),
            navigation=GraphNavigation(),
            filter_metadata={"subheading": subheading, "part": part_idx + 1, "total_parts": len(sub_chunks)}
        ))
    return emitted, counter


## Pass 4: End-to-End Processing & Chunk Generation

We iterate through all sections, applying:
- **Heading-Aware Micro-Splitting**: Breaks long sections by `section_header` elements into topic chunks.
- **Sentence-Boundary Windowing**: Enforces ~512 token budget with 2-sentence overlap for long text runs.
- **Lead-In Folding & Footnote Absorption**: Pre-table lead-ins and post-table footnotes are absorbed into table chunks.
- **Table Row-Grouping**: Repeated multi-tier column headers on 6–10 row windows.
- **Bidirectional Pointers**: `prev_chunk_id`, `next_chunk_id`, and `section_sibling_ids`.


In [ ]:
all_chunks: List[UnifiedChunk] = []
chunk_counter = 0

for b_idx, sec_data in tqdm(sorted(sections_map.items()), desc="Processing TOC Sections"):
    toc_info = sec_data["toc_info"]
    elements = sec_data["elements"]
    
    section_chunks: List[UnifiedChunk] = []
    current_text_run = ""
    current_subheading = ""
    last_narrative_context = ""
    current_pages = set()
    lead_in_text = ""

    i = 0
    while i < len(elements):
        elem = elements[i]
        modality = elem["modality"]
        page_no = elem["page_no"]
        raw = elem["raw_data"]
        label = elem.get("label", "")
        
        if modality == "text":
            text_str = raw.get("text", "").strip()
            if not text_str:
                i += 1
                continue
                
            is_subheading = (label in ("section_header", "title"))
            
            # If we hit a sub-heading and already have text, flush previous topic!
            if is_subheading:
                if current_text_run:
                    flushed, chunk_counter = emit_text_chunks(current_text_run, current_subheading, current_pages, b_idx, toc_info, chunk_counter)
                    section_chunks.extend(flushed)
                    last_narrative_context = current_text_run
                    current_text_run = ""
                    current_pages = set()
                current_subheading = text_str
                current_pages.add(page_no)
                i += 1
                continue
                
            # Check if this text immediately precedes a table (lead-in candidate)
            next_is_table = (i + 1 < len(elements) and elements[i+1]["modality"] == "table")
            if next_is_table and approximate_token_count(text_str) < 80 and not current_text_run:
                lead_in_text = text_str
                i += 1
                continue
                
            current_text_run = heal_cross_page_text(current_text_run, text_str)
            current_pages.add(page_no)
            
        elif modality == "table":
            context_source = current_text_run if current_text_run.strip() else last_narrative_context
            surrounding_context = get_surrounding_context_snippet(context_source)
            if current_text_run:
                flushed, chunk_counter = emit_text_chunks(current_text_run, current_subheading, current_pages, b_idx, toc_info, chunk_counter)
                section_chunks.extend(flushed)
                last_narrative_context = current_text_run
                current_text_run = ""
                current_pages = set()
                current_subheading = ""
                
            # Process Table: Inspect succeeding elements for footnotes
            absorbed_footnotes = []
            j = i + 1
            while j < len(elements):
                next_elem = elements[j]
                if next_elem["modality"] == "text":
                    nxt_txt = next_elem["raw_data"].get("text", "").strip()
                    if is_footnote_element(nxt_txt):
                        absorbed_footnotes.append(nxt_txt)
                        j += 1
                        continue
                break
            i = j - 1  # Advance past absorbed footnotes
            
            # Format Table
            header, rows = format_docling_table(raw)
            table_title = lead_in_text if lead_in_text else f"Table ({toc_info['active_title']})"
            lead_in_text = ""  # Reset lead-in
            
            # Row-group windowing (repeat headers)
            ROW_WINDOW = 10
            row_chunks = [rows[k:k+ROW_WINDOW] for k in range(0, max(len(rows), 1), ROW_WINDOW)]
            
            for r_idx, r_group in enumerate(row_chunks):
                chunk_counter += 1
                c_id = f"chunk_sec{b_idx:03d}_table_{chunk_counter:04d}_part{r_idx+1}"
                tbl_md = f"### {table_title} (Part {r_idx+1}/{len(row_chunks)})\n\n"
                if surrounding_context:
                    tbl_md += f"> **Surrounding Context**: \"{surrounding_context}\"\n\n"
                tbl_md += table_to_markdown(header, r_group)
                if absorbed_footnotes:
                    tbl_md += "\n\n**Inlined Footnotes:**\n" + "\n".join(f"- {fn}" for fn in absorbed_footnotes)
                    
                tok = approximate_token_count(tbl_md)
                citation = f"JPMC 2025 Annual Report, p. {page_no} ({' > '.join(toc_info['path'])})"
                
                section_chunks.append(UnifiedChunk(
                    chunk_id=c_id,
                    modality=ChunkModality.TABLE,
                    text=tbl_md,
                    token_count=tok,
                    char_count=len(tbl_md),
                    provenance=ChunkProvenance(
                        page_start=page_no,
                        page_end=page_no,
                        page_range=[page_no],
                        citation=citation,
                        bbox=elem.get("bbox")
                    ),
                    toc=TOCContext(
                        toc_section=toc_info["active_title"],
                        toc_parent=toc_info["parent"],
                        toc_path=toc_info["path"],
                        toc_level=toc_info["level"],
                        toc_bookmark_index=b_idx
                    ),
                    navigation=GraphNavigation(),
                    inlined_footnotes=absorbed_footnotes,
                    filter_metadata={"is_table_part": r_idx + 1, "total_table_parts": len(row_chunks), "surrounding_context": surrounding_context}
                ))
                
        elif modality == "picture":
            context_source = current_text_run if current_text_run.strip() else last_narrative_context
            surrounding_context = get_surrounding_context_snippet(context_source)
            # Lookup detailed VLM analytical summary
            pic_idx = elem.get("ref", "").split("/")[-1]
            cls_info = detailed_image_summaries.get(pic_idx) or existing_img_classifications.get(pic_idx, {})
            
            # Skip discarded/decorative icons
            if not cls_info.get("is_decorative") and cls_info.get("rag_action") != "discard":
                title = cls_info.get("title") or f"Visual Figure (Page {page_no})"
                category = cls_info.get("category") or "Diagram / Chart"
                detailed_sum = cls_info.get("detailed_summary") or cls_info.get("summary", "")
                takeaway = cls_info.get("business_takeaway", "")
                quant_data = cls_info.get("quantitative_data", [])
                tags = cls_info.get("tags", [category])
                img_path = cls_info.get("path")
                
                chunk_counter += 1
                c_id = f"chunk_sec{b_idx:03d}_figure_{chunk_counter:04d}"
                fig_text = f"### Visual Figure: {title} ({category}, Page {page_no})\n\n"
                if surrounding_context:
                    fig_text += f"> **Surrounding Context**: \"{surrounding_context}\"\n\n"
                fig_text += f"{detailed_sum}\n\n"
                if takeaway:
                    fig_text += f"**Key Business Takeaway**: {takeaway}\n\n"
                if quant_data:
                    fig_text += f"**Extracted Metrics**: {quant_data}\n\n"
                fig_text += f"**Context**: {toc_info['active_title']} ({' > '.join(toc_info['path'])})"
                
                tok = approximate_token_count(fig_text)
                citation = f"JPMC 2025 Annual Report, p. {page_no} ({' > '.join(toc_info['path'])})"
                
                section_chunks.append(UnifiedChunk(
                    chunk_id=c_id,
                    modality=ChunkModality.FIGURE,
                    text=fig_text,
                    token_count=tok,
                    char_count=len(fig_text),
                    provenance=ChunkProvenance(
                        page_start=page_no,
                        page_end=page_no,
                        page_range=[page_no],
                        citation=citation,
                        bbox=elem.get("bbox")
                    ),
                    toc=TOCContext(
                        toc_section=toc_info["active_title"],
                        toc_parent=toc_info["parent"],
                        toc_path=toc_info["path"],
                        toc_level=toc_info["level"],
                        toc_bookmark_index=b_idx
                    ),
                    navigation=GraphNavigation(),
                    multimodal=MultimodalAnchor(
                        image_path=img_path,
                        visual_category=category,
                        quantitative_metrics=[str(m) for m in quant_data] if isinstance(quant_data, list) else [str(quant_data)],
                        tags=tags
                    ),
                    filter_metadata={"visual_category": category, "surrounding_context": surrounding_context}
                ))
        i += 1
        
    # Flush remaining text run at section end
    if current_text_run:
        flushed, chunk_counter = emit_text_chunks(current_text_run, current_subheading, current_pages, b_idx, toc_info, chunk_counter)
        section_chunks.extend(flushed)
        
    # Link siblings within section
    sibling_ids = [c.chunk_id for c in section_chunks]
    for s_idx, c in enumerate(section_chunks):
        c.navigation.chunk_index_in_section = s_idx + 1
        c.navigation.total_chunks_in_section = len(section_chunks)
        c.navigation.section_sibling_ids = sibling_ids
        
    all_chunks.extend(section_chunks)

# Global forward/backward linked-list pointers across all chunks
for idx in range(len(all_chunks)):
    if idx > 0:
        all_chunks[idx].navigation.prev_chunk_id = all_chunks[idx - 1].chunk_id
    if idx < len(all_chunks) - 1:
        all_chunks[idx].navigation.next_chunk_id = all_chunks[idx + 1].chunk_id

print(f"Generated {len(all_chunks)} fully enriched, cross-linked chunks across all modalities!")


## Pass 5: Double Serialization & Validation

We serialize all chunks to:
1. `chunking/output/unified_chunks.jsonl`: One JSON object per line for streaming vector DB loaders.
2. `chunking/output/unified_chunks.json`: Full hierarchical JSON array for inspection.


In [ ]:
jsonl_out = OUTPUT_DIR / "unified_chunks.jsonl"
json_out = OUTPUT_DIR / "unified_chunks.json"

with open(jsonl_out, "w", encoding="utf-8") as f_jl:
    for c in all_chunks:
        f_jl.write(c.model_dump_json() + "\n")

with open(json_out, "w", encoding="utf-8") as f_j:
    json.dump([c.model_dump() for c in all_chunks], f_j, indent=2)

print(f"Saved {len(all_chunks)} chunks to:")
print(f"  - {jsonl_out} ({jsonl_out.stat().st_size / (1024*1024):.2f} MB)")
print(f"  - {json_out} ({json_out.stat().st_size / (1024*1024):.2f} MB)")

# Modality Breakdown Stats
from collections import Counter
modality_counts = Counter(c.modality.value for c in all_chunks)
print("\nModality Distribution:")
for mod, cnt in modality_counts.items():
    print(f"  - {mod.upper():15s}: {cnt:4d} chunks")


In [ ]:
print("Sample Inspection of Chunks across Modalities:\n")

for sample_mod in [ChunkModality.TEXT, ChunkModality.TABLE, ChunkModality.FIGURE]:
    matching = [c for c in all_chunks if c.modality == sample_mod]
    if matching:
        c = matching[0]
        print(f"============================================================")
        print(f"Modality: {c.modality.value.upper()} | ID: {c.chunk_id}")
        print(f"Citation: {c.provenance.citation}")
        print(f"TOC Path: {' > '.join(c.toc.toc_path)}")
        print(f"Prev Chunk: {c.navigation.prev_chunk_id} | Next Chunk: {c.navigation.next_chunk_id}")
        print(f"Tokens: {c.token_count} | Footnotes: {len(c.inlined_footnotes)}")
        print(f"--- Text Preview ---")
        preview = c.text[:300] + "..." if len(c.text) > 300 else c.text
        print(preview)
        print()
